[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfc/janus-core/blob/main/docs/source/tutorials/python/elasticity.ipynb)

# Elasticity

## Set up environment (optional)

These steps are required to run this tutorial with Google Colab. To do so, uncomment and run the cell below.

These instructions but may work for other systems too, but it is typically preferable to prepare a virtual environment separately before running this notebook if possible.

In [ ]:
# ! uv pip install git+https://github.com/WillBaldwin0/graph_electrostatics@v0.4.0
# ! uv pip install janus-core[mace,orb,chgnet,visualise] data-tutorials --system # Install janus-core with MACE, Orb, CHGNet, and WeasWidget, and data-tutorials


To ensure you have the latest version of `janus-core` installed, compare the output of the following cell to the latest version available at https://pypi.org/project/janus-core/

In [ ]:
from janus_core import __version__

print(__version__)

## Prepare data and modules

In [ ]:
from weas_widget import WeasWidget

from ase.build import bulk, nanotube
from ase.lattice.cubic import Diamond
from ase.io import read

import numpy as np
import matplotlib.pyplot as plt

from janus_core.calculations.elasticity import Elasticity

Use `data_tutorials` to get the data required for this tutorial:

### Generation of samples

In ```janus_core``` we can calculate the elasticity tensor via [pymatgen](https://github.com/materialsproject/pymatgen).

As an example we will do this for three samples: Aluminium, Diamond, and a Carbon-nanotube. Then compare their relative stiffnesses.

Using the ASE we can build each sample using the utility functions in ```ase.build```.

Firstly Aluminium can be generated using ```ase.build.bulk```.

In [ ]:
al = bulk("Al", crystalstructure="fcc")*(3,3,3)

v=WeasWidget()
v.from_ase(al)
v

Next we can build Diamond,

In [ ]:
diamond = Diamond('C')*(2,2,2)

v=WeasWidget()
v.from_ase(diamond)
v

And finally the Carbon-nanotube,

In [ ]:
nt = nanotube(6, 6, length=4)
# Place into a box
nt.cell = [nt.cell[2][2], nt.cell[2][2], nt.cell[2][2]]
nt.pbc = [True, True, True]
v=WeasWidget()
v.from_ase(nt)
v

### Elasticity calculations

```janus-core``` includes the elasticity calculation type for calculating the elasticity tensor. This is done by first creating a set of deformed (strained) structures. The stress tensor tensor ($\sigma$) is then calculated for each of these deformed structures. Finally the elasticity tensor $C^{ijkl}$ is estimated from the relationship between stress and strain. Or mathematically:

$$ \sigma^{ij} = C^{ijkl} E^{kl} $$

To find $C^{ijkl}$ we can use ```janus_core.calculations.elasticity.Elasticity```. First we construct the calculation object and run the calculation using the ```run()``` method. This will first geometry optimize the input structure, and then generate the set of deformed structures from it. Finally stresses are calculated for each deformed structure.

In [ ]:
elasticity_aluminium = Elasticity(
    struct=al.copy(),
    arch="mace_mp",
    device="cpu",
    model="small",
    calc_kwargs={"default_dtype": "float64"}
).run()

The results will be written to ```Al27-elastic_tensor.dat``` (automatically generated from our structures composition). The file includes the components of the $3\times3\times3\times3$ elasticity tensor in (row-major) Voigt reduced form ($6\times6$), which is preceded by various derived values such as the shear and bulk moduli.

In [ ]:
! cat janus_results/Al27-elastic_tensor.dat

The experimental bulk, shear, and Young's moduli are approximately 76 GPa, 26 GPa, and 68 GPa for Aluminium. Our results are in the right ball park.

For elasticity the main options for user control are the magnitudes of the applied shear and normal strains as well as their number.

By default 4 shear and 4 normal strains are applied. Split equally positively and negatively, and along each possible direction. This means $2\times 4 \times 3 = 24$ total stress calculations.

Each magnitude sets the *largest* strain applied, with the remaining strains spread evenly between $\pm$ that value. They should be kept small enough that the stress response stays linear: the elasticity tensor is fitted on the assumption that $\sigma^{ij} = C^{ijkl} E^{kl}$, so strains large enough to reach the anharmonic part of the energy surface return a secant slope rather than the zero-strain modulus. For Aluminium, normal strains beyond a few percent inflate the bulk modulus, and shear strains approaching the ideal shear strength inflate $C_{44}$ severalfold. Increasing `n_strains` does not rescue an over-wide range, it only dilutes it.

For example we can increase the shear and normal magnitudes modestly, and apply 16 strains of each type as follows:

In [ ]:
calc_elasticity_aluminium = Elasticity(
    struct=al.copy(),
    arch="mace_mp",
    device="cpu",
    model="small",
    calc_kwargs={"default_dtype": "float64"},
    shear_magnitude=0.1,
    normal_magnitude=0.02,
    n_strains=16
)
elasticity_aluminium = calc_elasticity_aluminium.run()

Using Weas we can get a feel for how the applied deformation are impacting our structure by playing through them like any other trajectory.

In this case, since we saved the calculation object we can directly access the ```deformed_structures```

In [ ]:
v=WeasWidget()
v.from_ase(calc_elasticity_aluminium.deformed_structures)
v

Also note that the deformed structure contain the applied strain within their ```info``` attributes

In [ ]:
s = calc_elasticity_aluminium.deformed_structures[-1].info["strain"]
print(f"(Voigt) Strain recorded in structure: {calc_elasticity_aluminium.strains[-1].voigt}")
print(f"Original strain {calc_elasticity_aluminium.strains[-1]}")

In [ ]:
elasticity_diamond = Elasticity(
    struct=diamond.copy(),
    arch="mace_mp",
    device="cpu",
    model="small",
    write_structures=True,
    calc_kwargs={"default_dtype": "float64"}
).run()

Notice for diamond we used the ```write_structures=True``` key word argument, in this case our results directory will include the file ```C64-elasticity-generated.extxyz``` and ```C64-elasticity-opt.extxyz```.

The latter is simply our initial diamond structure, but geometry optimized. The former is again the set of strained structures used to calculate the elasticity. We can read the file and observe the impact of the strains like so

In [ ]:
v=WeasWidget()
diamond_strained = read("janus_results/C64-elasticity-generated.extxyz", index=":")
v.from_ase(diamond_strained)
v

In [ ]:
elasticity_nt = Elasticity(
    struct=nt.copy(),
    arch="mace_mp",
    device="cpu",
    model="small",
    calc_kwargs={"default_dtype": "float64"}
).run()

In [ ]:
materials = ("Aluminium", "Diamond", "Carbon-nanotube")
tensors = (elasticity_aluminium, elasticity_diamond, elasticity_nt)
colours = ("#2a78d6", "#eb6834", "#1baf7a")

moduli = {
    "Bulk\n$K_V$": [t.k_voigt for t in tensors],
    "Shear\n$G_V$": [t.g_voigt for t in tensors],
    # Note https://github.com/materialsproject/pymatgen/issues/4435
    "Young's\n$E$": [t.y_mod / 1e9 for t in tensors],
}
values = np.array(list(moduli.values()))

y = np.arange(len(moduli))
height = 0.26

fig, ax = plt.subplots(figsize=(7.5, 4.2), layout="constrained")
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

for i, (material, colour) in enumerate(zip(materials, colours)):
    bars = ax.barh(y + (i - 1) * height, values[:, i], height=height * 0.92,
                   color=colour, label=material, zorder=3)
    # Label every bar, so the softest are still readable
    ax.bar_label(bars, fmt=" %.0f", padding=1, fontsize=9, color="#52514e")

ax.set_yticks(y, moduli.keys())
ax.invert_yaxis()
ax.set_xlabel("Modulus (GPa)")
ax.set_xlim(0, values.max() * 1.12)
ax.set_title("Elastic moduli", loc="left",
             color="#0b0b0b", fontsize=13, pad=34)

ax.xaxis.grid(True, color="#e1e0d9", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(colors="#898781", length=0)
for label in ax.get_yticklabels():
    label.set_color("#0b0b0b")
ax.xaxis.label.set_color("#52514e")
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")

ax.legend(frameon=False, ncols=3, loc="lower left",
          bbox_to_anchor=(0, 1.02), labelcolor="#52514e")